<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/09-etica-el-modelo-que-discrimina/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 2.9 · Ética II: el modelo que aprendió a discriminar

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Qué le pasa a un modelo cuando **aprende de una decisión sesgada**.
- Por qué **quitar una columna no alcanza**.
- Qué hacer **en lugar** de filtrar con un modelo.

## 📊 En Excel lo harías así

No hay fórmula para esto. Es una propuesta que puede llegar a tu oficina: *"tenemos miles de CV y años de decisiones: entrenemos un modelo que filtre a los postulantes"*.

Suena eficiente. Hoy vemos por qué no conviene.

> ⚠️ **Este notebook no sirve para filtrar personas, ni se debe adaptar para eso.** Solo muestra tasas de un archivo inventado, con un sesgo programado a propósito.

## 🐍 En Python

### Paso 1: lo que va a aprender

El objetivo es `entrevistado`: la decisión del reclutador. Fíjate bien: el modelo no aprende quién **trabaja bien**. Aprende **a quién llamaba el reclutador**, incluido el sesgo que vimos en el 2.8.

Entrenamos hasta 2024 y miramos 2025.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def mujeres_que_llamaria(entreno, prueba, columnas):
    """Solo cuenta: a cuántas mujeres llamaría un modelo que imita al reclutador, en cada cargo."""
    x_entreno = pd.get_dummies(entreno[columnas], dtype=int)
    x_prueba = pd.get_dummies(prueba[columnas], dtype=int).reindex(
        columns=x_entreno.columns, fill_value=0
    )
    modelo = make_pipeline(StandardScaler(), LogisticRegression())
    modelo.fit(x_entreno, entreno["entrevistado"])
    probabilidad = pd.Series(modelo.predict_proba(x_prueba)[:, 1], index=prueba.index)
    mujeres = 0
    for _, grupo in prueba.groupby("cargo"):
        cupo = int(grupo["entrevistado"].sum())
        orden = probabilidad[grupo.index].sort_values(ascending=False, kind="stable")
        mujeres += (prueba.loc[orden.index[:cupo], "sexo"] == "F").sum()
    return int(mujeres)

In [3]:
postulaciones = pd.read_csv(DATOS + "postulaciones.csv")

entreno = postulaciones[postulaciones["fecha"] < "2025-01-01"]
prueba = postulaciones[postulaciones["fecha"] >= "2025-01-01"]

base = ["cargo", "anios_experiencia", "puntaje_prueba"]
indicadores = ["meses_sin_empleo", "puede_viajar"]

sorteo = 0
for _, grupo in prueba.groupby("cargo"):
    sorteo += grupo["entrevistado"].sum() * (grupo["sexo"] == "F").mean()

print(len(prueba), prueba["entrevistado"].sum())
print(round(sorteo))
print(prueba.loc[prueba["sexo"] == "F", "entrevistado"].sum())

783 215
72
35


En 2025 se llamó a 215 personas de 783. Si dentro de cada cargo se **sorteara** a quién llamar, serían unas 72 mujeres. El reclutador llamó a 35. Esa es nuestra referencia.

### Paso 2: un modelo que imita

Si el modelo reemplaza al reclutador, ¿a cuántas mujeres llamaría? La función llama a tantas personas por cargo como hubo en 2025, las de mayor probabilidad según el modelo. **Solo devuelve una cuenta**, sin ninguna lista de personas.

In [4]:
mujeres_que_llamaria(entreno, prueba, base + indicadores + ["sexo"])

26

Unas 26: **menos que el reclutador**. ¿Por qué? El reclutador tiene algo de variación: a veces llama a alguien fuera de su patrón. Un corte del modelo aplica ese patrón **sin excepciones**. En estos datos, automatizar con un corte vuelve el sesgo más extremo.

Lo probamos con 30 versiones de estos datos, y pasó en las 30. Todas salen del mismo programa: es estable ante el azar, y no dice nada de otros supuestos. La comprobación está en `tests/test_temporada_2.py`.

### Paso 3: "quitemos la columna `sexo`"

Es lo primero que se le ocurre a cualquiera.

In [5]:
mujeres_que_llamaria(entreno, prueba, base + indicadores)

41

Unas 41, muy lejos de las 72. Sale algo mejor que el reclutador, pero eso es azar de estos datos: en la mayoría de nuestras 30 versiones queda igual o peor. Lo firme es que **en las 30 queda por debajo del sorteo**. El modelo ya no ve el sexo, pero ve los meses sin empleo y si puede viajar: **los indicadores indirectos del 2.8**.

### Paso 4: quitemos también esas

In [6]:
mujeres_que_llamaria(entreno, prueba, base)

72

Unas 72. ¿Resuelto? **No.** Este es el error más peligroso del episodio.

Aquí funcionó solo porque programamos dos indicadores, y sabemos cuáles son. Con datos reales hay muchos, y nadie te da la lista: el distrito, el colegio, una pausa en el CV, hasta cómo se redacta. **Borrar columnas no es una receta.**

Y aun así seguiría sin servir, por lo más importante:

- **La etiqueta sigue siendo la decisión sesgada.** Cuanto más imita al reclutador, mejor se ve en sus métricas. Sus aciertos solo dicen que se parece a él.
- **No sabemos cómo habría trabajado quien no fue llamado.** Es el mismo problema que vimos en el 1.7: el modelo de créditos solo ve a quienes ya recibieron crédito.
- **Contar mujeres no dice si se eligió bien.** Solo dice si hay una diferencia que explicar.

### Entonces, ¿qué hacer?

**No uses este modelo para filtrar postulantes.** En el 1.7 dijimos *"una persona decide"*, y sigue siendo cierto. Pero no basta, porque el sesgo salió de una persona. Hace falta:

1. **Definir los criterios antes** de ver a los postulantes, y escribirlos.
2. **Revisar las tasas por cargo** cada cierto tiempo, con la tabla dinámica del 2.8.
3. **Nombrar a un responsable** de esa revisión.
4. **Si existen datos de desempeño**, usarlos con mucho cuidado, porque también pueden tener sesgo.

> Las normas sobre empleo y datos personales se deben consultar en la fuente oficial. Este video no es asesoría legal.

## 🤖 Pídeselo a la IA

> *"Un proveedor me ofrece un sistema de IA para filtrar CV. Dame 8 preguntas concretas para hacerle antes de considerar comprarlo, sobre datos de entrenamiento, sesgo y responsabilidad."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Contar el resultado | `mujeres_que_llamaria(entreno, prueba, columnas)` |
| Quitar una columna | Dejarla fuera de la lista de `columnas` |
| Comparar con una referencia | El sorteo dentro de cada cargo |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/09-etica-el-modelo-que-discrimina/ejercicio.ipynb) y comprueba por tu cuenta lo que hace cada columna. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/09-etica-el-modelo-que-discrimina/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0